In [0]:
%sql
use catalog de_learning_workspace;
use schema silver;


In [0]:
from delta.tables import DeltaTable
def silver_incremental_load_table(table_name):
    last_modified = (
    spark.sql("""
        SELECT MAX(updated_at) AS updated_at
        FROM {table_name}
    """)
    .first()["updated_at"]
)

    if last_modified is None:
        last_modified = "1900-01-01 00:00:00"

    silver_source_df = spark.sql(f"""
    SELECT *
    FROM de_learning_workspace.bronze.{table_name}
    WHERE updated_at > TIMESTAMP('{last_modified}')
    """)



def silver_merge_table(table_name):
    target_table=DeltaTable.forName(f"de_learning_workspace.silver.{table_name}")
    (
        target_table.alias("t")
        .merge(
            silver_source_df.alias("s"),
            condition="t.customer_id = s.customer_id",
        )
        .whenMatchedUpdateAll(condition="s.updated_at>t.updated_at")
        .whenNotMatchedInsertAll()
        .execute()
    )




